In [ ]:
!pip install -q "timm>=1.0.9"

## 1. Configuración

In [ ]:
import os, io, json, time, zipfile
from collections import defaultdict
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from PIL import Image

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import timm


class CFG:
    DATA_DIR = "/kaggle/input/datasets/ibrahimagabardiop/niayes-crops-disease-v2"
    OUT_DIR  = "/kaggle/working"
    IMG_SIZE = 224

    EMB_CANDIDATOS  = ["vit_small_patch14_dinov2.lvd142m", "convnext_tiny.in12k_ft_in1k"] 
    EMB_MODEL       = None
    EMB_COSINE_MIN  = 0.98
    EMB_ORIENTACIONES = True    # 8 orientaciones del lado de entrenamiento
    EMB_BATCH       = 128
    EMB_CHUNK       = 256       # imagenes de evaluacion por bloque
    EMB_MAX_DESCARTE = 0.70     # freno de seguridad

    RUN_HASH       = True
    HASH_EXACTO    = 0          # umbral de Hamming; 0 = coincidencia exacta
    HASH_DIEDRAL_D3 = 3         

    N_EJEMPLOS = 6
    NUM_WORKERS = os.cpu_count() or 4
    SEED = 42

os.makedirs(CFG.OUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": 0.3})
print(f"Device: {device} | workers: {CFG.NUM_WORKERS}")

## 2. Particiones

In [ ]:
splits = {k: datasets.ImageFolder(os.path.join(CFG.DATA_DIR, k))
          for k in ("train", "val", "test")}
class_names = splits["train"].classes
num_classes = len(class_names)
for k, ds in splits.items():
    assert ds.classes == class_names, f"Las clases de {k} no coinciden"

paths  = {k: [p for p, _ in v.samples] for k, v in splits.items()}
labels = {k: np.array(v.targets) for k, v in splits.items()}
y_tr = labels["train"]

def short_name(raw):
    partes = raw.split("__")
    return partes[-1].replace("_", " ") if len(partes) > 1 else raw.replace("_", " ")

labels_cortos = [short_name(c) for c in class_names]
print(f"Clases: {num_classes}")
for k in ("train", "val", "test"):
    print(f"  {k:6s}: {len(paths[k]):>6d}")

## 3. Embeddings con orientaciones


In [ ]:
emb_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(CFG.IMG_SIZE), transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])


class ImagenesOrientadas(Dataset):
    # Devuelve (8, 3, H, W) si orientaciones=True; si no, (1, 3, H, W)
    def __init__(self, rutas, orientaciones):
        self.rutas, self.orientaciones = rutas, orientaciones

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, i):
        try:
            img = Image.open(self.rutas[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (CFG.IMG_SIZE, CFG.IMG_SIZE))
        t = emb_tf(img)
        if not self.orientaciones:
            return t.unsqueeze(0)
        vistas = []
        for base in (t, torch.flip(t, dims=[2])):          # original y reflejo
            for k in range(4):                             # 4 rotaciones de 90 grados
                vistas.append(torch.rot90(base, k, dims=[1, 2]).contiguous())
        return torch.stack(vistas)


@torch.no_grad()
def extraer(rutas, modelo, orientaciones):
    ds = ImagenesOrientadas(rutas, orientaciones)
    loader = DataLoader(ds, batch_size=max(1, CFG.EMB_BATCH // (8 if orientaciones else 1)),
                        shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    salida = []
    for lote in loader:
        b, v = lote.shape[:2]
        x = lote.view(b * v, *lote.shape[2:]).to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=device.type == "cuda"):
            e = modelo(x).float()
        salida.append(F.normalize(e, dim=1).view(b, v, -1).half().cpu())
    return torch.cat(salida)        # (N, V, D), normalizado


modelo_emb, ultimo_error = None, None
for nombre in CFG.EMB_CANDIDATOS:
    for kwargs in ({"img_size": CFG.IMG_SIZE}, {}):
        try:
            modelo_emb = timm.create_model(nombre, pretrained=True, num_classes=0,
                                           **kwargs).to(device).eval()
            CFG.EMB_MODEL = nombre
            break
        except Exception as e:
            ultimo_error = e
    if modelo_emb is not None:
        break
    print(f"  no disponible: {nombre} ({type(ultimo_error).__name__})")

if modelo_emb is None:
    raise RuntimeError(
        "No se pudo cargar ningún extractor de embeddings. Activa Internet en "
        "Settings, o adjunta los pesos como dataset. En esta versión el embedding "
        "es el detector principal, así que no se puede continuar sin él.")

print(f"Extractor: {CFG.EMB_MODEL}")
t0 = time.time()
E_train = extraer(paths["train"], modelo_emb, CFG.EMB_ORIENTACIONES)   # (N, 8, D)
E_val   = extraer(paths["val"],   modelo_emb, False)                   # (N, 1, D)
E_test  = extraer(paths["test"],  modelo_emb, False)
print(f"Embeddings en {time.time() - t0:.0f}s | train {tuple(E_train.shape)} "
      f"| val {tuple(E_val.shape)} | test {tuple(E_test.shape)}")
del modelo_emb; torch.cuda.empty_cache()

## 4. Vecinos más cercanos


In [ ]:
N_tr, V, D = E_train.shape
plano = E_train.view(N_tr * V, D).to(device)  


def pares_sobre(Eq, umbral, devolver_max=False):
    # Devuelve los pares (i_train, j_eval, similitud) por encima del umbral.
    # Con devolver_max=True, tambien la similitud maxima de cada j contra train.
    enc, maximos = [], np.zeros(Eq.shape[0], dtype=np.float32)
    q = Eq[:, 0].to(device)
    for ini in range(0, q.shape[0], CFG.EMB_CHUNK):
        bloque = q[ini:ini + CFG.EMB_CHUNK]
        sim = (bloque @ plano.T).view(bloque.shape[0], N_tr, V).amax(dim=2).float()
        maximos[ini:ini + bloque.shape[0]] = sim.max(dim=1).values.cpu().numpy()
        fila, col = (sim >= umbral).nonzero(as_tuple=True)
        for f, c in zip(fila.tolist(), col.tolist()):
            enc.append((c, ini + f, float(sim[f, c])))
        del sim
    return (enc, maximos) if devolver_max else enc


t0 = time.time()
pares_val,  max_val  = pares_sobre(E_val,  CFG.EMB_COSINE_MIN, devolver_max=True)
pares_test, max_test = pares_sobre(E_test, CFG.EMB_COSINE_MIN, devolver_max=True)
print(f"train↔val  : {len(pares_val)} pares")
print(f"train↔test : {len(pares_test)} pares")

# Duplicados internos: train contra si mismo, conservando la primera aparicion
internos, q_tr = set(), E_train[:, 0].to(device)
for ini in range(0, N_tr, CFG.EMB_CHUNK):
    bloque = q_tr[ini:ini + CFG.EMB_CHUNK]
    sim = (bloque @ plano.T).view(bloque.shape[0], N_tr, V).amax(dim=2).float()
    for f in range(bloque.shape[0]):
        i = ini + f
        anteriores = (sim[f, :i] >= CFG.EMB_COSINE_MIN).any().item() if i else False
        if anteriores:
            internos.add(i)
    del sim
print(f"duplicados internos en train: {len(internos)}")
print(f"Búsqueda completada en {time.time() - t0:.0f}s")
del plano, q_tr; torch.cuda.empty_cache()

### Barrido del umbral de coseno


In [ ]:
sim_val  = np.array([s for _, _, s in pares_val])
sim_test = np.array([s for _, _, s in pares_test])
todas = np.concatenate([sim_val, sim_test]) if len(sim_val) + len(sim_test) else np.array([])

barrido = []
for u in (0.95, 0.96, 0.97, 0.98, 0.99, 0.995, 0.999):
    nv = len({j for i, j, s in pares_val  if s >= u})
    nt = len({j for i, j, s in pares_test if s >= u})
    ntr = len({i for i, j, s in pares_val + pares_test if s >= u})
    barrido.append({"Umbral": u, "Val marcadas": nv, "% val": nv / len(paths["val"]),
                    "Test marcadas": nt, "% test": nt / len(paths["test"]),
                    "Train implicadas": ntr})
barrido = pd.DataFrame(barrido)
barrido.to_csv(os.path.join(CFG.OUT_DIR, "tabla_barrido_umbral.csv"), index=False)
print(barrido.round(4).to_string(index=False))
print(f"\nUmbral en uso: {CFG.EMB_COSINE_MIN}")

if len(todas):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(todas, bins=60, color="#2a6f97")
    ax.axvline(CFG.EMB_COSINE_MIN, ls="--", c="#c75146", lw=1.5,
               label=f"umbral = {CFG.EMB_COSINE_MIN}")
    ax.set_xlabel("Similitud coseno del par detectado"); ax.set_ylabel("Pares")
    ax.set_title("Distribución de similitudes por encima del umbral", fontweight="bold")
    ax.legend(fontsize=8)
    fig.savefig(os.path.join(CFG.OUT_DIR, "fig_distribucion_similitud.png")); plt.show()

## 5. El hash como diagnóstico


In [ ]:
def _dhash64(arr_u8):
    g = np.asarray(Image.fromarray(arr_u8).resize((9, 8), Image.BILINEAR), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def dhash_canonico(path):
    try:
        sq = np.asarray(Image.open(path).convert("L").resize((64, 64), Image.BILINEAR),
                        dtype=np.uint8)
    except Exception:
        return None
    mejor = None
    for base in (sq, np.fliplr(sq)):
        for k in range(4):
            h = _dhash64(np.ascontiguousarray(np.rot90(base, k)))
            if mejor is None or h < mejor:
                mejor = h
    return mejor


def _bands(h, n=4, bits=16):
    m = (1 << bits) - 1
    return [(i, (h >> (i * bits)) & m) for i in range(n)]


class HashIndex:
    def __init__(self, max_dist):
        self.max_dist, self.buckets = max_dist, defaultdict(list)
    def add(self, h, payload):
        if h is not None:
            for b in _bands(h):
                self.buckets[b].append((h, payload))
    def vecinos(self, h):
        if h is None:
            return []
        vistos, out = set(), []
        for b in _bands(h):
            for cand, p in self.buckets.get(b, ()):
                if p not in vistos and bin(cand ^ h).count("1") <= self.max_dist:
                    vistos.add(p); out.append(p)
        return out


pares_hash = {}
if CFG.RUN_HASH:
    t0 = time.time()
    with Pool(CFG.NUM_WORKERS) as pool:
        H = {k: pool.map(dhash_canonico, paths[k], chunksize=256)
             for k in ("train", "val", "test")}
    print(f"Hashes en {time.time() - t0:.0f}s")

    for etiqueta, dist in (("exacto", CFG.HASH_EXACTO), ("≤3 (v1)", CFG.HASH_DIEDRAL_D3)):
        idx = HashIndex(dist)
        for k in ("val", "test"):
            for j, h in enumerate(H[k]):
                idx.add(h, (k, j))
        enc = []
        for i, h in enumerate(H["train"]):
            for k, j in idx.vecinos(h):
                enc.append((i, k, j))
        pares_hash[etiqueta] = enc
        n_tr = len({i for i, _, _ in enc})
        print(f"  dHash diedral {etiqueta:8s}: {len(enc)} pares, "
              f"{n_tr} imágenes de entrenamiento implicadas")

In [ ]:
# Cuanto se equivocaba el hash: pares que marca y el embedding no respalda
emb_pares_set = {(i, "val", j) for i, j, _ in pares_val} | \
                {(i, "test", j) for i, j, _ in pares_test}

filas = []
for etiqueta, enc in pares_hash.items():
    s = set(enc)
    solo_hash = s - emb_pares_set
    contra = sum(1 for i, k, j in solo_hash if y_tr[i] != labels[k][j])
    filas.append({
        "Método": f"dHash diedral {etiqueta}",
        "Pares detectados": len(s),
        "Respaldados por embedding": len(s & emb_pares_set),
        "Solo hash": len(solo_hash),
        "% solo hash": len(solo_hash) / max(1, len(s)),
        "De esos, etiqueta distinta": contra,
        "% etiqueta distinta": contra / max(1, len(solo_hash)),
    })
filas.append({
    "Método": f"Embeddings (coseno ≥ {CFG.EMB_COSINE_MIN})",
    "Pares detectados": len(emb_pares_set),
    "Respaldados por embedding": len(emb_pares_set), "Solo hash": 0,
    "% solo hash": 0.0,
    "De esos, etiqueta distinta": sum(
        1 for i, k, j in emb_pares_set if y_tr[i] != labels[k][j]),
    "% etiqueta distinta": np.nan,
})
tabla_metodos = pd.DataFrame(filas)
tabla_metodos.to_csv(os.path.join(CFG.OUT_DIR, "tabla_dedup_metodos.csv"), index=False)
print(tabla_metodos.round(4).to_string(index=False))

print("\nUn porcentaje alto en «solo hash» con etiquetas distintas es la huella")
print("de los falsos positivos: el hash empareja hojas diferentes de clases")
print("diferentes. Esta tabla es la evidencia que justifica usar embeddings.")

## 6. Partición congelada

In [ ]:
fuga = {i for i, _, _ in pares_val} | {i for i, _, _ in pares_test}
internos_netos = internos - fuga
descartar = fuga | internos_netos
n_train = len(paths["train"])

frac = len(descartar) / n_train
if frac > CFG.EMB_MAX_DESCARTE:
    raise RuntimeError(
        f"Se descartaría el {frac:.1%} del entrenamiento, por encima del límite de "
        f"{CFG.EMB_MAX_DESCARTE:.0%}.\nRevisa el barrido de umbral y sube "
        f"EMB_COSINE_MIN antes de continuar.")

keep_idx = np.array(sorted(set(range(n_train)) - descartar), dtype=np.int64)
np.save(os.path.join(CFG.OUT_DIR, "keep_indices_train.npy"), keep_idx)

test_marcadas = {j for _, j, _ in pares_test}
val_marcadas  = {j for _, j, _ in pares_val}
clean_test = np.ones(len(paths["test"]), dtype=bool)
clean_test[list(test_marcadas)] = False
np.save(os.path.join(CFG.OUT_DIR, "clean_test_mask.npy"), clean_test)

print(f"Train original:         {n_train}")
print(f"  descartadas por fuga: {len(fuga)}  ({len(fuga)/n_train:.2%})")
print(f"  descartadas internas: {len(internos_netos)}  ({len(internos_netos)/n_train:.2%})")
print(f"Train congelado:        {len(keep_idx)}  ({len(keep_idx)/n_train:.2%})")
print(f"Test con duplicado:     {len(test_marcadas)} de {len(paths['test'])}  "
      f"({len(test_marcadas)/len(paths['test']):.2%})")
print(f"Test sin fuga:          {int(clean_test.sum())}")
print(f"Val con duplicado:      {len(val_marcadas)} de {len(paths['val'])}")

import hashlib
PARTICION_SHA = hashlib.sha256(np.ascontiguousarray(keep_idx).tobytes()).hexdigest()[:16]
print(f"\nHuella de la partición: {PARTICION_SHA}")

In [ ]:
counts = np.bincount(y_tr[keep_idx], minlength=num_classes)
orig = np.bincount(y_tr, minlength=num_classes)
dist = pd.DataFrame({"Clase": labels_cortos, "Train original": orig,
                     "Train congelado": counts, "Descartadas": orig - counts,
                     "% descartado": (orig - counts) / np.maximum(1, orig),
                     "Test total": np.bincount(labels["test"], minlength=num_classes)})
dist["Test con duplicado"] = [len(set(np.where(labels["test"] == c)[0]) & test_marcadas)
                              for c in range(num_classes)]
dist["% test contaminado"] = dist["Test con duplicado"] / np.maximum(1, dist["Test total"])
dist.sort_values("% test contaminado", ascending=False).to_csv(
    os.path.join(CFG.OUT_DIR, "tabla_fuga_por_clase.csv"), index=False)
print(dist.sort_values("% test contaminado", ascending=False).head(10).round(3).to_string(index=False))

criticas = dist[dist["Train congelado"] < 100]
if len(criticas):
    print(f"\nAVISO — {len(criticas)} clases quedan con menos de 100 imágenes:")
    print(criticas[["Clase", "Train original", "Train congelado"]].to_string(index=False))

d = dist.set_index("Clase").sort_values("Train original")
fig, ax = plt.subplots(figsize=(10, 11))
y = np.arange(len(d))
ax.barh(y, d["Train congelado"], color="#2a6f97", label="Entrenamiento congelado")
ax.barh(y, d["Descartadas"], left=d["Train congelado"], color="#c75146",
        label="Descartado (duplicado o fuga)")
ax.set_yticks(y); ax.set_yticklabels(d.index, fontsize=7)
ax.set_xlabel("Imágenes de entrenamiento")
ax.set_title("Filtro de duplicados por clase — detector por embeddings", fontweight="bold")
ax.legend(loc="lower right", fontsize=8); ax.grid(axis="y", visible=False)
fig.savefig(os.path.join(CFG.OUT_DIR, "fig_fuga_por_clase.png")); plt.show()

## 7. Verificación visual

In [ ]:
# Los pares mas debiles: si estos son copias reales, el umbral es correcto
ordenados = sorted(pares_test, key=lambda p: p[2])[:CFG.N_EJEMPLOS]
if ordenados:
    fig, axes = plt.subplots(len(ordenados), 2,
                             figsize=(6.5, 3.1 * len(ordenados)), squeeze=False)
    for fila, (i, j, s) in enumerate(ordenados):
        for col, (p, etq) in enumerate([
                (paths["train"][i], f"train · {short_name(class_names[y_tr[i]])}"),
                (paths["test"][j],  f"test · {short_name(class_names[labels['test'][j]])}")]):
            axes[fila, col].imshow(Image.open(p).convert("RGB"))
            axes[fila, col].set_title(etq, fontsize=8)
            axes[fila, col].set_xticks([]); axes[fila, col].set_yticks([])
            axes[fila, col].grid(False)
        axes[fila, 0].set_ylabel(f"cos={s:.3f}", fontsize=8, rotation=0,
                                 ha="right", va="center", labelpad=28)
    fig.suptitle("Pares detectados con MENOR similitud\n"
                 "si no son copias, hay que subir EMB_COSINE_MIN",
                 fontweight="bold", y=1.0)
    fig.savefig(os.path.join(CFG.OUT_DIR, "fig_pares_limite.png")); plt.show()
    print(f"Similitud del par más débil mostrado: {ordenados[0][2]:.4f}")

## 8. Informe

In [ ]:
contra = [(i, k, j) for i, k, j in emb_pares_set if y_tr[i] != labels[k][j]]
pd.DataFrame([{"train_idx": i, "train_clase": class_names[y_tr[i]],
               "train_path": paths["train"][i], "split": k, "eval_idx": j,
               "eval_clase": class_names[labels[k][j]], "eval_path": paths[k][j]}
              for i, k, j in contra]).to_csv(
    os.path.join(CFG.OUT_DIR, "pares_etiqueta_contradictoria.csv"), index=False)

reporte = {
    "version": "fase0-v2",
    "metodo_principal": {
        "tipo": "embeddings autosupervisados",
        "modelo": CFG.EMB_MODEL,
        "coseno_min": CFG.EMB_COSINE_MIN,
        "orientaciones_en_train": 8 if CFG.EMB_ORIENTACIONES else 1,
        "pares_revisados": ["train-val", "train-test", "train-train"],
    },
    "metodo_diagnostico": {
        "tipo": "dHash canónico diedral de 64 bits",
        "umbral_exacto": CFG.HASH_EXACTO, "umbral_v1": CFG.HASH_DIEDRAL_D3,
        "nota": ("Descartado como detector: produce falsos positivos sobre imágenes "
                 "de fondo uniforme. Canonizar por el mínimo de 8 orientaciones "
                 "reduce la entropía efectiva del hash."),
    },
    "train": {"original": n_train, "descartadas_por_fuga": len(fuga),
              "descartadas_internas": len(internos_netos), "congelado": int(len(keep_idx))},
    "val":  {"total": len(paths["val"]),  "con_duplicado_en_train": len(val_marcadas)},
    "test": {"total": len(paths["test"]), "con_duplicado_en_train": len(test_marcadas),
             "sin_fuga": int(clean_test.sum())},
    "comparacion_metodos": tabla_metodos.to_dict(orient="records"),
    "barrido_umbral": barrido.to_dict(orient="records"),
    "pares_totales": len(emb_pares_set),
    "pares_etiqueta_contradictoria": len(contra),
    "particion_sha256_16": PARTICION_SHA,
    "clases": class_names, "semilla": CFG.SEED,
}
with open(os.path.join(CFG.OUT_DIR, "dedup_report.json"), "w") as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False, default=float)

salidas = ["keep_indices_train.npy", "clean_test_mask.npy", "dedup_report.json",
           "tabla_dedup_metodos.csv", "tabla_fuga_por_clase.csv",
           "tabla_barrido_umbral.csv", "pares_etiqueta_contradictoria.csv",
           "fig_fuga_por_clase.png", "fig_pares_limite.png",
           "fig_distribucion_similitud.png"]
with zipfile.ZipFile(os.path.join(CFG.OUT_DIR, "fase0_dedup_v2.zip"), "w",
                     zipfile.ZIP_DEFLATED) as z:
    for f_ in salidas:
        p = os.path.join(CFG.OUT_DIR, f_)
        if os.path.exists(p):
            z.write(p, f_)

print(json.dumps({k: v for k, v in reporte.items()
                  if k not in ("clases", "comparacion_metodos", "barrido_umbral")},
                 indent=2, ensure_ascii=False))
print("\nRevisa fig_pares_limite.png ANTES de crear el dataset.")
print("Si los pares más débiles no son copias, sube EMB_COSINE_MIN y reejecuta.")